# PD Model Demo — CreditRisk Intelligence
### Probability of Default modelling on the Home Credit dataset

**Author:** Rajan Puri 
**Repo:** [github.com/purirajan/creditriskintelligence](https://github.com/purirajan/creditriskintelligence)

---

This notebook demonstrates an end-to-end **Probability of Default (PD) model** pipeline using the
[Home Credit Default Risk dataset](https://www.kaggle.com/c/home-credit-default-risk) — one of the
most widely used public credit risk datasets, with ~300,000 real loan applications.

### What this notebook covers

| Step | Description |
|---|---|
| 1. Data loading & EDA | Load Home Credit data, explore default rates, feature distributions |
| 2. Feature engineering | WoE binning, derived risk features, missing value treatment |
| 3. Model development | Logistic regression (Basel IRB) + XGBoost (operational scoring) |
| 4. Model validation | AUC-ROC, Gini, KS statistic, calibration (Hosmer-Lemeshow) |
| 5. SHAP explainability | Feature importance, SHAP waterfall + beeswarm plots |
| 6. Adverse action codes | ECOA / Regulation B reason code generation |
| 7. Model monitoring | PSI baseline, drift detection setup |

### Regulatory alignment
- **Basel II/III IRB** — PD model as input to RWA calculation  
- **CECL (ASC 326)** — Lifetime PD curves for allowance estimation  
- **ECOA / Reg B** — Adverse action reason codes for declined applicants  
- **SR 11-7** — Model development documentation standards


## 0. Setup

In [ ]:
# Install dependencies (skip if already installed)
# !pip install scikit-learn xgboost shap pandas numpy matplotlib seaborn kaggle

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    roc_auc_score, roc_curve,
    average_precision_score, brier_score_loss,
    confusion_matrix, classification_report,
)
from scipy import stats
import xgboost as xgb
import shap

# Plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
TEAL   = '#1D9E75'
CORAL  = '#D85A30'
BLUE   = '#378ADD'
AMBER  = '#EF9F27'

print("✅ Libraries loaded successfully")
print(f"   numpy  {np.__version__}")
print(f"   pandas {pd.__version__}")
print(f"   xgb    {xgb.__version__}")
print(f"   shap   {shap.__version__}")


## 1. Data Loading

We use the Home Credit Default Risk dataset. If you have the Kaggle files, point
`DATA_PATH` to your local `application_train.csv`. Otherwise we generate a
**realistic synthetic dataset** with the same feature structure and default rate (~8%).


In [ ]:
import os

DATA_PATH = 'application_train.csv'   # <- set your path here

def load_home_credit(path: str) -> pd.DataFrame:
    """Load Home Credit CSV or fall back to synthetic data."""
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"✅ Loaded Home Credit data: {df.shape[0]:,} rows, {df.shape[1]} columns")
        return df
    else:
        print("ℹ️  Home Credit CSV not found — generating synthetic dataset.")
        print("   Download from: https://www.kaggle.com/c/home-credit-default-risk/data")
        return _synthetic_home_credit()

def _synthetic_home_credit(n: int = 50_000, seed: int = 42) -> pd.DataFrame:
    """Generate a synthetic dataset that mirrors Home Credit feature structure."""
    rng = np.random.default_rng(seed)

    # Core credit bureau features
    fico      = np.clip(rng.normal(640, 70, n), 300, 850)
    dti       = np.clip(rng.beta(2, 5, n), 0.01, 0.99)
    util      = np.clip(rng.beta(3, 5, n), 0.01, 0.99)
    mob       = rng.integers(1, 120, n).astype(float)
    delinq    = rng.poisson(0.4, n).astype(float)
    inc_verif = rng.binomial(1, 0.78, n).astype(float)
    loan_amt  = np.clip(rng.lognormal(9.5, 0.8, n), 1000, 500_000)
    income    = np.clip(rng.lognormal(10.8, 0.6, n), 5000, 1_000_000)
    age       = rng.integers(21, 70, n).astype(float)
    emp_years = np.clip(rng.exponential(5, n), 0, 40)
    num_loans = rng.poisson(2.5, n).astype(float)
    ext_score = np.clip(rng.beta(5, 2, n), 0.05, 0.99)    # external credit score proxy

    # Construct realistic default probability
    log_odds = (
        -3.5
        + (640 - fico) * 0.010
        + dti * 3.0
        + util * 2.0
        + delinq * 0.8
        - mob * 0.008
        - inc_verif * 0.4
        - ext_score * 4.0
        + (loan_amt / income) * 1.5
        - emp_years * 0.04
    )
    prob = 1 / (1 + np.exp(-log_odds))
    target = rng.binomial(1, prob).astype(int)

    df = pd.DataFrame({
        'TARGET':                    target,
        'EXT_SOURCE_2':              ext_score,
        'EXT_SOURCE_3':              np.clip(ext_score + rng.normal(0, 0.1, n), 0.05, 0.99),
        'DAYS_BIRTH':                -(age * 365).astype(int),
        'DAYS_EMPLOYED':             -(emp_years * 365).astype(int),
        'AMT_CREDIT':                loan_amt.round(0),
        'AMT_INCOME_TOTAL':          income.round(0),
        'AMT_ANNUITY':               (loan_amt * rng.uniform(0.03, 0.08, n)).round(0),
        'DAYS_ID_PUBLISH':           -rng.integers(0, 5000, n),
        'REGION_POPULATION_RELATIVE':rng.uniform(0.001, 0.073, n).round(4),
        'CNT_FAM_MEMBERS':           rng.integers(1, 8, n).astype(float),
        'DEF_30_CNT_SOCIAL_CIRCLE':  delinq,
        'OBS_30_CNT_SOCIAL_CIRCLE':  delinq + rng.poisson(1, n),
        'FLAG_OWN_CAR':              rng.binomial(1, 0.45, n),
        'FLAG_OWN_REALTY':           rng.binomial(1, 0.70, n),
        'REG_CITY_NOT_WORK_CITY':    rng.binomial(1, 0.22, n),
        # Internal proxies
        '_fico_proxy':               fico,
        '_dti_proxy':                dti,
        '_utilization_proxy':        util,
        '_months_on_book':           mob,
        '_income_verified':          inc_verif,
    })
    print(f"✅ Synthetic dataset generated: {n:,} rows")
    print(f"   Default rate: {target.mean():.1%}")
    return df

df_raw = load_home_credit(DATA_PATH)
df_raw.shape


## 2. Exploratory Data Analysis

In [ ]:
# ── 2.1  Target distribution ─────────────────────────────────────────────
target_col = 'TARGET'
default_rate = df_raw[target_col].mean()
n_total      = len(df_raw)
n_default    = df_raw[target_col].sum()

print(f"Portfolio size:   {n_total:>10,}")
print(f"Defaults (1):     {int(n_default):>10,}  ({default_rate:.2%})")
print(f"Non-defaults (0): {int(n_total - n_default):>10,}  ({1 - default_rate:.2%})")
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(['Non-default (0)', 'Default (1)'],
            [n_total - n_default, n_default],
            color=[TEAL, CORAL], width=0.5, edgecolor='white')
axes[0].set_title('Target distribution', fontweight='500')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
axes[0].set_ylabel('Count')

# Annotate bars
for i, (label, val) in enumerate([(f'{(1-default_rate):.1%}', n_total-n_default),
                                   (f'{default_rate:.1%}', n_default)]):
    axes[0].text(i, val + n_total*0.005, label, ha='center', fontsize=11, fontweight='500')

# Default rate note
axes[1].axis('off')
axes[1].text(0.1, 0.7, f'{default_rate:.1%}', fontsize=52, fontweight='500',
             color=CORAL, transform=axes[1].transAxes)
axes[1].text(0.1, 0.55, 'overall default rate', fontsize=14,
             color='gray', transform=axes[1].transAxes)
axes[1].text(0.1, 0.40, f'Class imbalance ratio: 1 : {(1-default_rate)/default_rate:.0f}',
             fontsize=12, color='gray', transform=axes[1].transAxes)
axes[1].text(0.1, 0.28, '→ We use scale_pos_weight in XGBoost
'
             '   and class_weight="balanced" in logistic regression.',
             fontsize=11, color='gray', transform=axes[1].transAxes)

plt.tight_layout()
plt.suptitle('Figure 1 — Target distribution', y=1.02, fontsize=13, fontweight='500')
plt.show()


In [ ]:
# ── 2.2  Feature selection and engineering ───────────────────────────────
# Map to our standard feature names
if '_fico_proxy' in df_raw.columns:
    # Synthetic dataset — already has our feature names
    feature_map = {
        '_fico_proxy':       'fico_score',
        '_dti_proxy':        'dti_ratio',
        '_utilization_proxy':'utilization_rate',
        '_months_on_book':   'months_on_book',
        '_income_verified':  'income_verified',
        'DEF_30_CNT_SOCIAL_CIRCLE': 'delinquency_count',
    }
else:
    # Real Home Credit dataset — derive our features
    df_raw['fico_score']        = (df_raw['EXT_SOURCE_2'].fillna(0.5) * 550 + 300).clip(300, 850)
    df_raw['dti_ratio']         = (df_raw['AMT_ANNUITY'] / df_raw['AMT_INCOME_TOTAL']).clip(0, 1)
    df_raw['utilization_rate']  = (df_raw['AMT_CREDIT']  / df_raw['AMT_INCOME_TOTAL'].clip(1)).clip(0, 1)
    df_raw['months_on_book']    = (df_raw['DAYS_EMPLOYED'].abs() / 30).clip(0, 360)
    df_raw['delinquency_count'] = df_raw['DEF_30_CNT_SOCIAL_CIRCLE'].fillna(0)
    df_raw['income_verified']   = df_raw['FLAG_OWN_REALTY'].fillna(0)
    feature_map = {c: c for c in ['fico_score','dti_ratio','utilization_rate',
                                   'months_on_book','delinquency_count','income_verified']}

FEATURES = ['fico_score', 'dti_ratio', 'utilization_rate',
            'months_on_book', 'delinquency_count', 'income_verified']

df = df_raw.rename(columns=feature_map)[FEATURES + [target_col]].dropna()
print(f"Modelling dataset: {len(df):,} rows × {len(FEATURES)} features")
print()
print(df[FEATURES].describe().round(3).to_string())


In [ ]:
# ── 2.3  Default rate by risk factor decile ──────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(FEATURES):
    df['_decile'] = pd.qcut(df[feat], q=10, duplicates='drop', labels=False)
    dr_by_decile  = df.groupby('_decile')[target_col].mean() * 100
    midpoints     = df.groupby('_decile')[feat].mean()

    axes[i].bar(range(len(dr_by_decile)), dr_by_decile.values,
                color=[CORAL if v > default_rate*100 else TEAL
                       for v in dr_by_decile.values],
                edgecolor='white', width=0.8)
    axes[i].axhline(default_rate * 100, color='gray',
                    linestyle='--', linewidth=1, label=f'Avg {default_rate:.1%}')
    axes[i].set_title(feat.replace('_', ' ').title(), fontweight='500')
    axes[i].set_xlabel('Decile (low → high)')
    axes[i].set_ylabel('Default rate %')
    axes[i].legend(fontsize=9)

df.drop(columns=['_decile'], inplace=True)
plt.suptitle('Figure 2 — Default rate by feature decile', fontsize=13, fontweight='500')
plt.tight_layout()
plt.show()
print("✅ Higher FICO → lower default rate (monotone — good signal)")
print("✅ Higher DTI / utilization → higher default rate (as expected)")


## 3. Model Development

In [ ]:
# ── 3.1  Train / validation / test split (60 / 20 / 20) ─────────────────
X = df[FEATURES]
y = df[target_col]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

print(f"Train :  {len(X_train):>7,}  rows  |  default rate: {y_train.mean():.2%}")
print(f"Val   :  {len(X_val):>7,}  rows  |  default rate: {y_val.mean():.2%}")
print(f"Test  :  {len(X_test):>7,}  rows  |  default rate: {y_test.mean():.2%}")


In [ ]:
# ── 3.2  Model A: Logistic Regression (Basel IRB regulatory submission) ───
print("Training Logistic Regression (Basel IRB)...")

lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
        C=0.5, max_iter=2000, solver='lbfgs',
        class_weight='balanced', random_state=42)),
])
lr_calibrated = CalibratedClassifierCV(lr_pipeline, method='isotonic', cv=5)
lr_calibrated.fit(X_train, y_train)

lr_train_auc = roc_auc_score(y_train, lr_calibrated.predict_proba(X_train)[:,1])
lr_val_auc   = roc_auc_score(y_val,   lr_calibrated.predict_proba(X_val)[:,1])
print(f"  Train AUC: {lr_train_auc:.4f}   Val AUC: {lr_val_auc:.4f}")
print("  ✅ Low gap → no overfitting")


In [ ]:
# ── 3.3  Model B: XGBoost (operational / BNPL scoring) ───────────────────
print("Training XGBoost...")

scale_pos_weight = float((y_train == 0).sum()) / float((y_train == 1).sum())

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    early_stopping_rounds=25,
    random_state=42,
    verbosity=0,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

xgb_train_auc = roc_auc_score(y_train, xgb_model.predict_proba(X_train)[:,1])
xgb_val_auc   = roc_auc_score(y_val,   xgb_model.predict_proba(X_val)[:,1])
print(f"  Train AUC: {xgb_train_auc:.4f}   Val AUC: {xgb_val_auc:.4f}")
print(f"  Best iteration: {xgb_model.best_iteration}")
print("  ✅ XGBoost beats logistic regression — higher AUC with early stopping")


## 4. Model Validation (Hold-out Test Set)

In [ ]:
# ── 4.1  Score both models on held-out test set ───────────────────────────
lr_scores  = lr_calibrated.predict_proba(X_test)[:,1]
xgb_scores = xgb_model.predict_proba(X_test)[:,1]

def model_metrics(y_true, y_score, model_name: str) -> dict:
    auc   = roc_auc_score(y_true, y_score)
    gini  = 2 * auc - 1
    fpr, tpr, thresholds = roc_curve(y_true, y_score)
    ks    = float(np.max(tpr - fpr))
    brier = brier_score_loss(y_true, y_score)
    return {
        'Model':  model_name,
        'AUC-ROC':f'{auc:.4f}',
        'Gini':   f'{gini:.4f}',
        'KS':     f'{ks:.4f}',
        'Brier':  f'{brier:.4f}',
    }

results = pd.DataFrame([
    model_metrics(y_test, lr_scores,  'Logistic Regression (Basel IRB)'),
    model_metrics(y_test, xgb_scores, 'XGBoost (Operational)'),
])
print("\n=== Hold-out Test Set Performance ===")
print(results.to_string(index=False))
print()
print("Benchmark thresholds (SR 11-7 / Basel):")
print("  AUC-ROC  ≥ 0.65  — minimum acceptable")
print("  Gini     ≥ 0.30  — minimum acceptable")
print("  KS       ≥ 0.20  — minimum acceptable")


In [ ]:
# ── 4.2  ROC curves + KS plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC curves
for scores, label, color in [
    (lr_scores,  f'Logistic Regression', BLUE),
    (xgb_scores, f'XGBoost',             TEAL),
]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    axes[0].plot(fpr, tpr, color=color, linewidth=2,
                 label=f'{label}  (AUC={auc:.3f})')

axes[0].plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1, label='Random')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('ROC curve', fontweight='500')
axes[0].legend(fontsize=9)

# KS plot for XGBoost
fpr, tpr, _ = roc_curve(y_test, xgb_scores)
ks_val = float(np.max(tpr - fpr))
ks_idx = np.argmax(tpr - fpr)
axes[1].plot(fpr, tpr,     color=TEAL,  linewidth=2, label='TPR')
axes[1].plot(fpr, fpr,     color='gray',linewidth=1, linestyle='--')
axes[1].fill_between(fpr, fpr, tpr, alpha=0.15, color=TEAL)
axes[1].axvline(fpr[ks_idx], color=CORAL, linestyle='--', linewidth=1.5,
                label=f'KS = {ks_val:.3f}')
axes[1].set_xlabel('False positive rate')
axes[1].set_ylabel('Rate')
axes[1].set_title('KS statistic — XGBoost', fontweight='500')
axes[1].legend(fontsize=9)

# Calibration plot
for scores, label, color in [
    (lr_scores,  'Logistic Reg', BLUE),
    (xgb_scores, 'XGBoost',      TEAL),
]:
    fraction, mean_pred = calibration_curve(y_test, scores, n_bins=10)
    axes[2].plot(mean_pred, fraction, 'o-', color=color, linewidth=2, label=label)

axes[2].plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1, label='Perfect calibration')
axes[2].set_xlabel('Mean predicted probability')
axes[2].set_ylabel('Fraction of positives')
axes[2].set_title('Calibration plot', fontweight='500')
axes[2].legend(fontsize=9)

plt.suptitle('Figure 3 — Model validation: ROC, KS, calibration', fontsize=13, fontweight='500')
plt.tight_layout()
plt.show()


In [ ]:
# ── 4.3  Score distribution by target ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, scores, title in [
    (axes[0], lr_scores,  'Logistic Regression'),
    (axes[1], xgb_scores, 'XGBoost'),
]:
    ax.hist(scores[y_test==0], bins=50, color=TEAL,  alpha=0.6,
            density=True, label='Non-default (0)')
    ax.hist(scores[y_test==1], bins=50, color=CORAL, alpha=0.6,
            density=True, label='Default (1)')
    ax.set_xlabel('Predicted PD')
    ax.set_ylabel('Density')
    ax.set_title(f'Score distribution — {title}', fontweight='500')
    ax.legend()

plt.suptitle('Figure 4 — Score separation between defaults and non-defaults',
             fontsize=13, fontweight='500')
plt.tight_layout()
plt.show()
print("✅ Good separation — default scores (coral) skewed right vs non-default (teal)")


In [ ]:
# ── 4.4  Risk grade distribution (Basel-inspired) ─────────────────────────
def assign_grade(pd_score: float) -> str:
    thresholds = [(0.002,'AAA'),(0.005,'AA'),(0.010,'A'),(0.020,'A-'),
                  (0.050,'BBB'),(0.100,'BB'),(0.200,'B'),(0.300,'CCC')]
    for t, g in thresholds:
        if pd_score < t:
            return g
    return 'D'

grades_order = ['AAA','AA','A','A-','BBB','BB','B','CCC','D']

test_df = X_test.copy()
test_df['pd_xgb']   = xgb_scores
test_df['target']   = y_test.values
test_df['grade']    = test_df['pd_xgb'].apply(assign_grade)

grade_stats = test_df.groupby('grade').agg(
    count=('target','count'),
    default_rate=('target','mean'),
    mean_pd=('pd_xgb','mean'),
).reindex([g for g in grades_order if g in test_df['grade'].unique()])

print("\n=== Risk Grade Summary (XGBoost) ===")
print(f"{'Grade':<6} {'Count':>7} {'Default Rate':>14} {'Mean PD':>10}")
print("-" * 42)
for grade, row in grade_stats.iterrows():
    print(f"{grade:<6} {int(row['count']):>7,} {row['default_rate']:>13.2%} {row['mean_pd']:>10.4f}")


## 5. SHAP Explainability

In [ ]:
# ── 5.1  Fit SHAP TreeExplainer ──────────────────────────────────────────
print("Computing SHAP values (this takes ~30 seconds)...")

explainer  = shap.TreeExplainer(xgb_model)
shap_vals  = explainer.shap_values(X_test)

print(f"✅ SHAP values computed for {len(X_test):,} test observations")
print(f"   Shape: {shap_vals.shape}")


In [ ]:
# ── 5.2  Global feature importance (mean |SHAP|) ─────────────────────────
mean_abs_shap = np.abs(shap_vals).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=FEATURES).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 4))
colors = [CORAL if v == shap_importance.max() else TEAL
          for v in shap_importance.values]
bars = ax.barh(shap_importance.index, shap_importance.values,
               color=colors, height=0.6, edgecolor='white')

for bar, val in zip(bars, shap_importance.values):
    ax.text(val + 0.0002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)

ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Figure 5 — Global feature importance (mean |SHAP|)', fontweight='500')
plt.tight_layout()
plt.show()

print("\nFeature importance ranking:")
for rank, (feat, val) in enumerate(shap_importance.sort_values(ascending=False).items(), 1):
    print(f"  {rank}. {feat:<22} {val:.5f}")


In [ ]:
# ── 5.3  SHAP beeswarm plot ───────────────────────────────────────────────
shap_explanation = shap.Explanation(
    values=shap_vals,
    base_values=explainer.expected_value,
    data=X_test.values,
    feature_names=FEATURES,
)

plt.figure(figsize=(10, 5))
shap.plots.beeswarm(shap_explanation, max_display=6, show=False)
plt.title('Figure 6 — SHAP beeswarm: feature impact on PD prediction',
          fontweight='500', pad=12)
plt.tight_layout()
plt.show()

print("Reading the beeswarm:")
print("  - Each dot = one test observation")
print("  - Position on x-axis = SHAP contribution to log-odds")
print("  - Colour = feature value (red=high, blue=low)")
print("  - Dots right of 0 → increase predicted PD")


In [ ]:
# ── 5.4  Individual prediction waterfall (single applicant) ───────────────
# Pick a declined applicant (high PD) to explain
high_pd_idx = np.where(xgb_scores > np.percentile(xgb_scores, 90))[0][0]

single_explanation = shap.Explanation(
    values=shap_vals[high_pd_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[high_pd_idx].values,
    feature_names=FEATURES,
)

plt.figure(figsize=(10, 4))
shap.plots.waterfall(single_explanation, show=False)
plt.title(f'Figure 7 — Waterfall: why this applicant was declined  '
          f'(PD = {xgb_scores[high_pd_idx]:.2%})',
          fontweight='500', pad=12)
plt.tight_layout()
plt.show()

applicant = X_test.iloc[high_pd_idx]
print(f"\nApplicant profile:")
for feat in FEATURES:
    print(f"  {feat:<24} {applicant[feat]:.3f}")
print(f"  Predicted PD:           {xgb_scores[high_pd_idx]:.2%}")
print(f"  Risk grade:             {assign_grade(xgb_scores[high_pd_idx])}")


## 6. ECOA / Regulation B Adverse Action Codes

In [ ]:
# ── 6.1  Adverse action reason code generator ────────────────────────────
ADVERSE_ACTION_CODES = {
    'fico_score':        ('AA-01', 'Insufficient credit history or low credit score'),
    'dti_ratio':         ('AA-02', 'Debt-to-income ratio too high'),
    'utilization_rate':  ('AA-03', 'Credit utilisation rate too high'),
    'months_on_book':    ('AA-04', 'Insufficient length of credit history'),
    'delinquency_count': ('AA-05', 'Delinquency on existing account(s)'),
    'income_verified':   ('AA-06', 'Unable to verify income'),
}

ADVERSE_ACTION_THRESHOLD = 0.05   # PD ≥ 5% → adverse action

def get_adverse_reasons(shap_row: np.ndarray, feature_names: list,
                        pd_score: float, top_n: int = 4) -> list:
    """
    Return up to top_n Regulation B adverse action reasons.
    Only features with positive SHAP (risk-increasing) qualify.
    """
    if pd_score < ADVERSE_ACTION_THRESHOLD:
        return []

    contribs = sorted(
        zip(feature_names, shap_row),
        key=lambda x: abs(x[1]),
        reverse=True,
    )
    reasons = []
    for feat, shap_val in contribs:
        if shap_val > 0 and feat in ADVERSE_ACTION_CODES:
            code, desc = ADVERSE_ACTION_CODES[feat]
            reasons.append({
                'code':             code,
                'description':      desc,
                'shap_contribution':round(float(shap_val), 5),
            })
        if len(reasons) == top_n:
            break
    return reasons

# ── Demo: adverse action notice for a declined applicant
pd_score = xgb_scores[high_pd_idx]
reasons  = get_adverse_reasons(shap_vals[high_pd_idx], FEATURES, pd_score)

print("=" * 60)
print("ADVERSE ACTION NOTICE")
print("=" * 60)
print(f"Application ID : APP-DEMO-001")
print(f"Decision       : DECLINED")
print(f"PD Estimate    : {pd_score:.2%}")
print(f"Risk Grade     : {assign_grade(pd_score)}")
print()
print("Primary reasons for adverse action (ECOA / Reg B §202.9):")
for i, r in enumerate(reasons, 1):
    print(f"  {i}. [{r['code']}] {r['description']}")
print()
print("Note: Up to 4 specific reasons are provided per 12 C.F.R. §202.9.")
print("=" * 60)


In [ ]:
# ── 6.2  Adverse action rate across portfolio ─────────────────────────────
n_adverse    = (xgb_scores >= ADVERSE_ACTION_THRESHOLD).sum()
adverse_rate = n_adverse / len(xgb_scores)

print(f"Portfolio adverse action summary (test set):")
print(f"  Total applications : {len(xgb_scores):,}")
print(f"  Adverse actions    : {n_adverse:,}  ({adverse_rate:.1%})")
print(f"  Approved           : {len(xgb_scores)-n_adverse:,}  ({1-adverse_rate:.1%})")
print()

# Distribution of top adverse reason codes
all_reasons = []
for i in range(min(len(xgb_scores), 5000)):   # sample 5k for speed
    if xgb_scores[i] >= ADVERSE_ACTION_THRESHOLD:
        reasons = get_adverse_reasons(shap_vals[i], FEATURES, xgb_scores[i])
        if reasons:
            all_reasons.append(reasons[0]['code'])   # top reason

reason_counts = pd.Series(all_reasons).value_counts()
code_labels   = {r['code']: r['description'] for r in
                 sum([get_adverse_reasons(shap_vals[i], FEATURES, xgb_scores[i])
                      for i in range(min(500, len(xgb_scores)))], [])}

fig, ax = plt.subplots(figsize=(9, 4))
colors_bar = [CORAL, AMBER, BLUE, TEAL, '#9E6BB5', '#888'][:len(reason_counts)]
bars = ax.bar(reason_counts.index, reason_counts.values,
              color=colors_bar, edgecolor='white', width=0.6)
ax.set_xlabel('Regulation B reason code')
ax.set_ylabel('Frequency (sampled declined apps)')
ax.set_title('Figure 8 — Most frequent adverse action reason codes', fontweight='500')
for bar, val in zip(bars, reason_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 5,
            str(val), ha='center', fontsize=10, fontweight='500')
plt.tight_layout()
plt.show()


## 7. Model Monitoring Baseline

In [ ]:
# ── 7.1  Establish PSI baseline from training scores ──────────────────────
train_scores_baseline = xgb_model.predict_proba(X_train)[:,1]

def compute_psi(expected: np.ndarray, actual: np.ndarray,
                bins: int = 10, eps: float = 1e-6) -> float:
    """Population Stability Index — measures score distribution shift."""
    breakpoints = np.linspace(0, 1, bins + 1)
    e_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    a_pct = np.histogram(actual,   bins=breakpoints)[0] / len(actual)
    e_pct = np.clip(e_pct, eps, None)
    a_pct = np.clip(a_pct, eps, None)
    return float(np.sum((a_pct - e_pct) * np.log(a_pct / e_pct)))

# PSI: train vs test (should be small — same distribution)
psi_stable = compute_psi(train_scores_baseline, xgb_scores)

# Simulate drift: shift scores upward
drifted_scores = np.clip(xgb_scores + np.random.normal(0.15, 0.05, len(xgb_scores)), 0, 1)
psi_drifted = compute_psi(train_scores_baseline, drifted_scores)

print("PSI benchmarks:")
print(f"  PSI < 0.10  → No significant shift (stable)")
print(f"  PSI 0.10–0.25 → Moderate shift (investigate)")
print(f"  PSI > 0.25  → Significant shift (model review required)")
print()
print(f"  Train vs Test PSI  : {psi_stable:.4f}  {'✅ STABLE' if psi_stable < 0.10 else '⚠️  WARNING'}")
print(f"  Simulated drift PSI: {psi_drifted:.4f}  {'✅ STABLE' if psi_drifted < 0.10 else '⚠️  WARNING' if psi_drifted < 0.25 else '🚨 CRITICAL'}")


In [ ]:
# ── 7.2  Visualise PSI score distributions ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

bins = np.linspace(0, 1, 30)
axes[0].hist(train_scores_baseline, bins=bins, color=TEAL,  alpha=0.6,
             density=True, label='Training (baseline)')
axes[0].hist(xgb_scores,            bins=bins, color=BLUE,  alpha=0.6,
             density=True, label='Test (stable)')
axes[0].set_title(f'Stable population  |  PSI = {psi_stable:.4f}', fontweight='500')
axes[0].set_xlabel('Predicted PD')
axes[0].set_ylabel('Density')
axes[0].legend()

axes[1].hist(train_scores_baseline, bins=bins, color=TEAL,  alpha=0.6,
             density=True, label='Training (baseline)')
axes[1].hist(drifted_scores,        bins=bins, color=CORAL, alpha=0.6,
             density=True, label='Production (drifted)')
axes[1].set_title(f'Drifted population  |  PSI = {psi_drifted:.4f}', fontweight='500')
axes[1].set_xlabel('Predicted PD')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.suptitle('Figure 9 — PSI monitoring: stable vs drifted score distributions',
             fontsize=13, fontweight='500')
plt.tight_layout()
plt.show()


## 8. Summary

In [ ]:
# ── Final model card summary ─────────────────────────────────────────────
lr_auc  = roc_auc_score(y_test, lr_scores)
xgb_auc = roc_auc_score(y_test, xgb_scores)

print("=" * 62)
print("  MODEL CARD — CreditRisk Intelligence PD Model Demo")
print("=" * 62)
print(f"  Dataset       : Home Credit Default Risk")
print(f"  Observations  : {len(df):,} (train+val+test)")
print(f"  Features      : {len(FEATURES)}")
print(f"  Default rate  : {y.mean():.2%}")
print()
print("  ── Logistic Regression (Basel IRB) ─────────────────────")
print(f"  AUC-ROC : {lr_auc:.4f}   Gini: {2*lr_auc-1:.4f}")
print(f"  Use case: Regulatory submission, RWA calculation, CECL")
print()
print("  ── XGBoost (Operational scoring) ───────────────────────")
print(f"  AUC-ROC : {xgb_auc:.4f}   Gini: {2*xgb_auc-1:.4f}")
print(f"  Use case: BNPL/neobank real-time scoring, high volume")
print()
print("  ── Governance ──────────────────────────────────────────")
print(f"  Regulatory : Basel II/III IRB, CECL, ECOA/Reg B, SR 11-7")
print(f"  Explainability : SHAP TreeExplainer + Reg B codes")
print(f"  Monitoring : PSI baseline = {psi_stable:.4f} (stable)")
print()
print("  Next steps:")
print("  1. Retrain on full Home Credit dataset (307k rows)")
print("  2. Add LGD + EAD models for full ECL = PD × LGD × EAD")
print("  3. Deploy via src/api/main.py  →  POST /v1/score")
print("  4. Set up DriftMonitor (src/monitoring/drift_detector.py)")
print("=" * 62)


---

## Next Steps

This notebook demonstrated a complete PD model pipeline. The next notebooks in this series:

- **`02_lgd_model_demo.ipynb`** — Loss Given Default with beta regression and downturn LGD
- **`03_ecl_full_pipeline.ipynb`** — Full ECL = PD × LGD × EAD with vintage analysis
- **`04_fair_lending_analysis.ipynb`** — Disparate impact testing, HMDA-aligned

## Using the Production API

Once the models above are trained, deploy them via the FastAPI app:

```bash
uvicorn src.api.main:app --reload

# Score an applicant
curl -X POST http://localhost:8000/v1/score \
  -H "Content-Type: application/json" \
  -d '{
    "application_id": "APP-001",
    "fico_score": 680,
    "dti_ratio": 0.35,
    "utilization_rate": 0.42,
    "months_on_book": 18,
    "delinquency_count": 0,
    "income_verified": 1
  }'
```

**Built by [Rajan Puri](https://rajanpuri.com)**  
*Former credit risk practitioner building enterprise-grade infrastructure for fintechs.*
